# imports

In [2]:
import math
import json
import urllib
import datetime
import ephem
import firebase_admin
from firebase_admin import credentials, firestore
from firebase_client import get_firestore_client, clear_grb_alerts


/Users/halla.d/anaconda3/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.13) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


# get location from MPC code


In [3]:
"""
MPC Observatory Code Lookup
============================
Returns longitude and latitude for a given IAU/MPC observatory code.
"""

MPC_API_URL = "https://data.minorplanetcenter.net/api/obscodes"


def _parallax_to_lat(rho_cos_phi: float, rho_sin_phi: float) -> float:
    """
    Convert MPC parallax constants to geodetic latitude (degrees).

    The parallax constants give *geocentric* cylindrical coordinates:
        rho·cos(φ')  = distance from Earth's axis / equatorial radius
        rho·sin(φ')  = height above equatorial plane / equatorial radius

    We first recover the geocentric latitude, then apply the WGS-84
    flattening correction to get the geodetic latitude.
    """
    geocentric_lat_rad = math.atan2(rho_sin_phi, rho_cos_phi)

    # WGS-84 parameters
    f = 1 / 298.257223563  # flattening
    e2 = 2 * f - f**2  # first eccentricity squared

    # Iterative conversion geocentric → geodetic (converges in 2-3 steps)
    lat_rad = geocentric_lat_rad
    for _ in range(5):
        sin_lat = math.sin(lat_rad)
        N = 1 / math.sqrt(1 - e2 * sin_lat**2)
        rho = math.sqrt(rho_cos_phi**2 + rho_sin_phi**2)
        # altitude above ellipsoid (in earth radii) – small, but needed
        h = rho - N * math.sqrt(1 - e2 * sin_lat**2)
        lat_rad = math.atan2(
            rho_sin_phi + e2 * N * sin_lat,
            rho_cos_phi,
        )

    return math.degrees(lat_rad)


def mpc_api_lookup(code: str) -> dict:
    """
    Look up an observatory code via the official MPC REST API.

    Returns a dict with:
        longitude  – degrees east of Greenwich  (float)
        latitude   – geodetic latitude in degrees (float, derived if not in API)
        name       – observatory name (str)
        raw        – the full API response dict

    Raises ValueError if the code is not found.
    """
    payload = json.dumps({"obscode": code.upper()}).encode()
    req = urllib.request.Request(
        MPC_API_URL,
        data=payload,
        headers={"Content-Type": "application/json"},
        method="GET",
    )
    with urllib.request.urlopen(req, timeout=10) as resp:
        data = json.loads(resp.read())

    if not data:
        raise ValueError(f"Observatory code '{code}' not found in MPC database.")

    lon = float(data.get("longitude"))
    rho_cos = float(data.get("rhocosphi"))
    rho_sin = float(data.get("rhosinphi"))
    name = data.get("name", "")

    # Derive geodetic latitude from parallax constants
    lat = _parallax_to_lat(rho_cos, rho_sin) if (rho_cos and rho_sin) else None

    return {
        "code": code.upper(),
        "longitude": lon,
        "latitude": lat,
        "name": name,
        "raw": data,
    }

In [4]:
# test it
mpc_info = mpc_api_lookup("M44")
mpc_info

{'code': 'M44',
 'longitude': 54.92031,
 'latitude': 24.219594760614754,
 'name': 'Al-Khatim Observatory, Abu Dhabi ',
 'raw': {'created_at': 'Mon, 01 Feb 2021 00:51:12 GMT',
  'firstdate': '2021-01-22',
  'lastdate': None,
  'longitude': '54.92031',
  'name': 'Al-Khatim Observatory, Abu Dhabi ',
  'name_latex': 'Al-Khatim Observatory, Abu Dhabi ',
  'name_utf8': 'Al-Khatim Observatory, Abu Dhabi ',
  'obscode': 'M44',
  'observations_type': 'optical',
  'old_names': None,
  'rhocosphi': '0.912502',
  'rhosinphi': '0.407722',
  'short_name': 'Al-Khatim Observatory, Abu Dhabi ',
  'updated_at': 'Wed, 17 Sep 2025 18:12:06 GMT',
  'uses_two_line_observations': False,
  'web_link': None}}

# get sunset time

In [5]:
def get_sunset(
    lat: float,
    lon: float,
    date: datetime.date | None = None,
    elevation_m: float = 0.0,
) -> dict:
    """
    Return sunset and sunrise times for an observer at (lat, lon).

    Parameters
    ----------
    lat          : latitude  in decimal degrees  (+N / -S)
    lon          : longitude in decimal degrees  (+E / -W)
                   Accepts both 0-360 east and -180..+180 conventions.
    date         : date to compute for; defaults to today (UTC)
    elevation_m  : elevation above sea level in metres (optional, default 0)

    Returns
    -------
    dict with keys:
        sunset_utc    – datetime (UTC, timezone-aware)
        sunrise_utc   – datetime (UTC, timezone-aware)
        transit_utc   – solar noon (UTC, timezone-aware)
        sunset_local  – sunset in the observer's mean solar time
        polar_day     – True if sun never sets
        polar_night   – True if sun never rises
    """
    if date is None:
        date = datetime.date.today()

    # Normalise longitude to -180..+180
    lon_norm = lon if lon <= 180 else lon - 360

    obs = ephem.Observer()
    obs.lat = str(lat)
    obs.lon = str(lon_norm)
    obs.elevation = elevation_m
    obs.date = date.strftime("%Y/%m/%d 12:00:00")  # start at local noon

    sun = ephem.Sun()

    def _to_aware_dt(ephem_date) -> datetime.datetime:
        """Convert an ephem Date to a timezone-aware UTC datetime."""
        return datetime.datetime.strptime(str(ephem_date), "%Y/%m/%d %H:%M:%S").replace(
            tzinfo=datetime.timezone.utc
        )

    result = {
        "date": date,
        "latitude": lat,
        "longitude": lon_norm,
        "polar_day": False,
        "polar_night": False,
    }

    try:
        result["sunrise_utc"] = _to_aware_dt(obs.previous_rising(sun))
        result["sunset_utc"] = _to_aware_dt(obs.next_setting(sun))
        result["transit_utc"] = _to_aware_dt(obs.next_transit(sun))
    except ephem.AlwaysUpError:
        result["polar_day"] = True
        result["sunrise_utc"] = None
        result["sunset_utc"] = None
        result["transit_utc"] = None
    except ephem.NeverUpError:
        result["polar_night"] = True
        result["sunrise_utc"] = None
        result["sunset_utc"] = None
        result["transit_utc"] = None

    # Mean local solar time = UTC + lon/15 hours
    if result["sunset_utc"]:
        offset = datetime.timedelta(hours=lon_norm / 15)
        result["sunset_local"] = result["sunset_utc"] + offset
        result["sunrise_local"] = result["sunrise_utc"] + offset
    else:
        result["sunset_local"] = None
        result["sunrise_local"] = None

    return result

In [6]:
# test it
sunset_info = get_sunset(mpc_info["latitude"], mpc_info["longitude"])
sunset_utc = sunset_info["sunset_utc"]

# get alerts for a given date


In [7]:
def get_alerts_for_date(date = datetime.date.today()) -> list[dict]:

    """
    Retrieve all GRB alerts from Firestore for a given date object.

    Args:
        date: Date object, default = today's date.

    Returns:
        A list of alert dictionaries matching the specified date.
    """

    #connect to db
    db = get_firestore_client()
    date_str = date.strftime("%y%m%d")  # e.g. "260317"
    
    # ── Query ───────────────────────────────────────────────────────────────
    docs = (
        db.collection("grb_alerts")
        .where("date_code", "==", date_str)
        .stream()
    )
    
    alerts = [doc.to_dict() for doc in docs]
    
    # ── Print results ────────────────────────────────────────────────────────
    print(f"Found {len(alerts)} alert(s) for {date}")
    # for alert in alerts:
    #     print(alert)
    
    return alerts

In [10]:
# test
yest_date = (datetime.date.today() - datetime.timedelta(days=1))
filtered_alerts = get_alerts_for_date(yest_date)

Found 4 alert(s) for 2026-03-16


In [14]:
import pandas as pd
pd.DataFrame(filtered_alerts)

,error_unit,dec_deg,altitude_degrees,ra_unit,trigger_date,snr,flux,ra,trigger_offset_value,trigger_offset_relation,...,ra_deg,error,email_time,obs_end_time_utc,magnitude,trigger_offset_unit,space_telescope,error_deg,fluence,flux_unit
0,arcmin,NaN,None,None,None,None,None,None,3.76,after_trigger,...,NaN,3.0,17:46:14,16:37:50.536,22.0,hours,EP,0.050000,None,None
1,arcsec,NaN,None,None,None,None,None,None,1.63,after_trigger,...,NaN,20.0,17:12:28,None,23.2,hours,EP,0.005556,None,None
2,None,NaN,None,None,2026-03-16,None,None,None,NaN,None,...,NaN,NaN,17:06:07,None,NaN,None,FERMI,NaN,None,None
3,deg,-79.3,None,deg,2026-03-16,None,None,305.5,NaN,None,...,305.5,3.2,17:23:26,None,NaN,None,FERMI,3.200000,None,None


In [ ]:
# continously check, if time now is 5 minutes before sunset, perform some logic
import time

while True:
    now_utc = datetime.datetime.now(datetime.timezone.utc)
    if (
        sunset_utc
        and (sunset_utc - now_utc) <= datetime.timedelta(minutes=5)
        and (sunset_utc - now_utc).days == 0
    ):
        print("Sunset is in 5 minutes!")
        break
    time.sleep(60)  # check every minute

KeyboardInterrupt: 